# Huấn luyện từng mô hình CNN

Notebook này giữ luồng như phiên bản ban đầu: chuẩn bị dữ liệu → chọn và huấn luyện một mô hình → vẽ loss/accuracy → lưu checkpoint tốt nhất.

Để chạy mô hình khác, đổi `MODEL_NAME` trong cell cấu hình rồi chạy lại các cell từ đầu. Notebook không tự chạy cả năm mô hình cùng lúc.

In [ ]:
from pathlib import Path
import sys
import json
import random
import time

# Xác định thư mục project trước khi import các file Python của project.
PROJECT_DIR = Path.cwd()
if not (PROJECT_DIR / 'baseline_cnn.py').is_file():
    candidate = PROJECT_DIR / 'fashion-product-classifier'
    if (candidate / 'baseline_cnn.py').is_file():
        PROJECT_DIR = candidate
    else:
        raise FileNotFoundError('Không tìm thấy thư mục project fashion-product-classifier.')
sys.path.insert(0, str(PROJECT_DIR))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from PIL import Image
from sklearn.model_selection import train_test_split
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

from models import MODEL_NAMES, count_parameters, create_model
from train import create_optimizer
from train_val import train_one_epoch, validate


CSV_PATH = PROJECT_DIR / 'clean_dataset.csv'
IMAGE_DIR = PROJECT_DIR / 'training_images'
OUTPUT_DIR = PROJECT_DIR / 'checkpoints'
MODEL_NAME = 'baseline'  # baseline, improved, resnet18, efficientnet_b0, resnet18_se
EPOCHS = 5
BATCH_SIZE = 32
LEARNING_RATE = 1e-3
USE_PRETRAINED = True
SEED = 42

if MODEL_NAME not in MODEL_NAMES:
    raise ValueError(f'MODEL_NAME phải là một trong: {MODEL_NAMES}')
if not CSV_PATH.is_file():
    raise FileNotFoundError(f'Không tìm thấy CSV: {CSV_PATH}')
if not IMAGE_DIR.is_dir():
    raise FileNotFoundError(f'Không tìm thấy thư mục ảnh: {IMAGE_DIR}')

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print('Mô hình:', MODEL_NAME)
print('Project:', PROJECT_DIR)

## 1. Đọc dữ liệu và tạo train/validation

In [ ]:
class FashionDataset(Dataset):
    def __init__(self, dataframe, image_dir, class_to_idx, transform):
        self.dataframe = dataframe.reset_index(drop=True)
        self.image_dir = Path(image_dir)
        self.class_to_idx = class_to_idx
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]
        image_path = self.image_dir / f"{row['id']}.jpg"
        with Image.open(image_path) as image:
            image = image.convert('RGB')
        image = self.transform(image)
        label = self.class_to_idx[row['articleType']]
        return image, label

df = pd.read_csv(CSV_PATH).dropna(subset=['id', 'articleType'])
df['articleType'] = df['articleType'].astype(str)
df = df.drop_duplicates(subset='id').reset_index(drop=True)
classes = sorted(df['articleType'].unique().tolist())
class_to_idx = {name: index for index, name in enumerate(classes)}
if len(classes) < 2:
    raise ValueError('Cần ít nhất hai lớp để huấn luyện.')

counts = df['articleType'].value_counts()
if (counts < 2).any():
    raise ValueError(f'Mỗi lớp cần ít nhất hai ảnh để chia train/validation: {counts.to_dict()}')

missing_images = [IMAGE_DIR / f"{image_id}.jpg" for image_id in df['id']
                  if not (IMAGE_DIR / f"{image_id}.jpg").is_file()]
if missing_images:
    raise FileNotFoundError(f'Thiếu {len(missing_images)} ảnh; ví dụ: {missing_images[:3]}')

train_df, val_df = train_test_split(
    df, test_size=0.2, random_state=SEED, stratify=df['articleType']
)
imagenet_mean = (0.485, 0.456, 0.406)
imagenet_std = (0.229, 0.224, 0.225)
train_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
val_transform = transforms.Compose([
    transforms.Resize((128, 128)),
    transforms.ToTensor(),
    transforms.Normalize(imagenet_mean, imagenet_std),
])
train_dataset = FashionDataset(train_df, IMAGE_DIR, class_to_idx, train_transform)
val_dataset = FashionDataset(val_df, IMAGE_DIR, class_to_idx, val_transform)
generator = torch.Generator().manual_seed(SEED)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, generator=generator)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)

print(f'Dữ liệu: {len(train_dataset)} train / {len(val_dataset)} validation')
print('Các lớp:', ', '.join(classes))

## 2. Khởi tạo và huấn luyện mô hình đã chọn

In [ ]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = create_model(
    MODEL_NAME, num_classes=len(classes), pretrained=USE_PRETRAINED
).to(device)
total_params, trainable_params = count_parameters(model)
optimizer = create_optimizer(model, MODEL_NAME, LEARNING_RATE, weight_decay=1e-4)
criterion = torch.nn.CrossEntropyLoss()
history = {'train_loss': [], 'train_accuracy': [], 'val_loss': [], 'val_accuracy': [], 'epoch_seconds': []}
best_val_accuracy = -1.0
best_epoch = 0
checkpoint_path = OUTPUT_DIR / f'{MODEL_NAME}_best.pth'
metadata_path = OUTPUT_DIR / f'{MODEL_NAME}_best.json'

print(f'Đang dùng: {device}')
print(f'Số tham số: {total_params:,} (train được: {trainable_params:,})')
start_time = time.perf_counter()
for epoch in range(1, EPOCHS + 1):
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    epoch_start = time.perf_counter()
    train_loss, train_accuracy = train_one_epoch(
        model, train_loader, criterion, optimizer, device
    )
    val_loss, val_accuracy = validate(model, val_loader, criterion, device)
    if device.type == 'cuda':
        torch.cuda.synchronize(device)
    epoch_seconds = time.perf_counter() - epoch_start

    history['train_loss'].append(train_loss)
    history['train_accuracy'].append(train_accuracy)
    history['val_loss'].append(val_loss)
    history['val_accuracy'].append(val_accuracy)
    history['epoch_seconds'].append(epoch_seconds)

    saved = ''
    if val_accuracy > best_val_accuracy:
        best_val_accuracy = val_accuracy
        best_epoch = epoch
        torch.save(model.state_dict(), checkpoint_path)
        metadata_path.write_text(json.dumps({
            'model_name': MODEL_NAME,
            'classes': classes,
            'class_to_idx': class_to_idx,
            'best_epoch': best_epoch,
            'best_val_accuracy': best_val_accuracy,
            'num_classes': len(classes),
        }, ensure_ascii=False, indent=2), encoding='utf-8')
        saved = ' | đã lưu checkpoint tốt nhất'

    print(
        f'Epoch {epoch}/{EPOCHS} | '
        f'Train loss: {train_loss:.4f}, acc: {train_accuracy:.4f} | '
        f'Val loss: {val_loss:.4f}, acc: {val_accuracy:.4f} | '
        f'{epoch_seconds:.1f}s{saved}'
    )

if device.type == 'cuda':
    torch.cuda.synchronize(device)
total_seconds = time.perf_counter() - start_time
history_df = pd.DataFrame(history)
history_df.insert(0, 'epoch', range(1, EPOCHS + 1))
history_df.insert(0, 'model', MODEL_NAME)
history_path = OUTPUT_DIR / f'{MODEL_NAME}_notebook_metrics.csv'
history_df.to_csv(history_path, index=False)

# Giữ bảng tích lũy để có thể so sánh sau khi chạy các model lần lượt.
summary_path = OUTPUT_DIR / 'notebook_results.csv'
summary_row = pd.DataFrame([{
    'model': MODEL_NAME,
    'parameters': total_params,
    'trainable_parameters': trainable_params,
    'epochs': EPOCHS,
    'best_epoch': best_epoch,
    'best_val_accuracy': best_val_accuracy,
    'best_val_loss': history['val_loss'][best_epoch - 1],
    'seconds_to_best': sum(history['epoch_seconds'][:best_epoch]),
    'total_training_seconds': total_seconds,
    'checkpoint': str(checkpoint_path),
}])
if summary_path.exists():
    previous = pd.read_csv(summary_path)
    previous = previous[previous['model'] != MODEL_NAME]
    summary_row = pd.concat([previous, summary_row], ignore_index=True)
summary_row.to_csv(summary_path, index=False)

print('Checkpoint tốt nhất:', checkpoint_path, f'(epoch {best_epoch})')
print(f'Thời gian huấn luyện: {total_seconds:.1f}s')

## 3. Vẽ biểu đồ train/validation

In [ ]:
epochs = range(1, EPOCHS + 1)
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs, history['train_loss'], 'o-', label='Train')
axes[0].plot(epochs, history['val_loss'], 'o-', label='Validation')
axes[0].set(title=f'{MODEL_NAME} - Loss', xlabel='Epoch', ylabel='Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)
axes[1].plot(epochs, history['train_accuracy'], 'o-', label='Train')
axes[1].plot(epochs, history['val_accuracy'], 'o-', label='Validation')
axes[1].set(title=f'{MODEL_NAME} - Accuracy', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

display(pd.read_csv(summary_path).sort_values('best_val_accuracy', ascending=False))